## Load libraries

In [ ]:
import polars as pl
import os
from mpl_toolkits.mplot3d import Axes3D
from sklearn.preprocessing import StandardScaler
import polars as pl
pl.Config.set_tbl_rows(20)  # show up to 20 rows
import polars.selectors as cs
import seaborn as sns
sns.set_style("darkgrid")  # darkgrid
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd

In [ ]:
def csv_to_parquet(path):
    os.makedirs("raw_data", exist_ok=True)
    filename = os.path.basename(path)
    parquet_filename = filename.replace(".csv", ".parquet")
    parquet_path = os.path.join(".\\raw_data", parquet_filename)
    # lazy scan (does NOT load everything into RAM)
    lf = pl.scan_csv(
        path,
        infer_schema_length=1000,
        ignore_errors=True
    )
    print(parquet_path)
    # stream directly to parquet
    lf.sink_parquet(
        parquet_path,
        compression="snappy"
    )
    print(f"Saved Parquet file to: {parquet_path}")

### Convert pandas to polar

In [ ]:
has_parquet = any(f.endswith(".parquet") for f in os.listdir("raw_data"))

if has_parquet:
    print("Parquet file already exists in data folder")
else:
    print("No parquet file found. Converting CSV → Parquet...")
    csv_to_parquet("./raw_data/accepted_2007_to_2018Q4.csv")

### Read accepted_df

In [ ]:
# Never use CSV again
accepted_df = pl.read_parquet("./raw_data/accepted_2007_to_2018Q4.parquet")
accepted_df.head()

In [ ]:
print(accepted_df.columns)
print(accepted_df.select('issue_d'))

In [ ]:
import polars as pl

# schema info
schema = accepted_df.schema

# null counts (computed once)
null_counts = accepted_df.null_count().row(0)

# total rows
n_rows = accepted_df.height

# build summary dataframe efficiently
type_summary = pl.DataFrame({
    "column": accepted_df.columns,
    "dtype": [str(schema[c]) for c in accepted_df.columns],
    "nulls": list(null_counts),
})

# compute percentages
type_summary = (
    type_summary
    .with_columns(
        (
            pl.col("nulls") / n_rows * 100
        ).round(2).alias("null_pct")
    )
    .sort("dtype")
)

print(f"Dataset shape: {accepted_df.shape}")

print("\n----- Null Values Distribution -----")

# avoid printing gigantic dataframe
print(type_summary)

## Data understanding

In [ ]:
# count nulls only
n_null_issue = (
    accepted_df
    .select(pl.col("issue_d").is_null().sum())
    .item()
)

print(f"Rows with null issue_d: {n_null_issue}")

# drop nulls directly
accepted_df = accepted_df.drop_nulls(subset=["issue_d"])

print("Remove null in column issue_d done!")
print(f"New shape: {accepted_df.shape}")

In [ ]:
# Polar String display
pl.Config.set_fmt_str_lengths(1000)

# Loan Status Distribution
print(f'Total Rows    : {accepted_df.shape[0]:,}')
print(f'Total Columns : {accepted_df.shape[1]}')
date_range = accepted_df.select(
    pl.col("issue_d")
      .str.strptime(pl.Date, "%b-%Y", strict=False)
      .min()
      .alias("min_date"),
    pl.col("issue_d")
      .str.strptime(pl.Date, "%b-%Y", strict=False)
      .max()
      .alias("max_date"),
)

print(
    f"Date Range    : "
    f"{date_range[0, 'min_date']:%b-%Y} → "
    f"{date_range[0, 'max_date']:%b-%Y}"
)

print('\n----- Loan Status Distribution -----')
print(accepted_df['loan_status'].value_counts().sort("count", descending=True))

## Data cleaning

In [ ]:
accepted_df["installment"].value_counts()

In [ ]:
# Keep only known outcomes
accepted_df = accepted_df.filter(
    pl.col("loan_status").is_in(["Fully Paid", "Charged Off"])
)

# Create binary target
accepted_df = accepted_df.with_columns(
    pl.when(pl.col("loan_status") == "Charged Off")
    .then(1)
    .otherwise(0)
    .cast(pl.Int8)
    .alias("target")
)

# Compute default rate
default_rate = (
    accepted_df
    .select(pl.col("target").mean())
    .item()
    * 100
)

print(f"Dataset columns: {accepted_df.columns}")
print(f"Overall Default Rate: {default_rate:.2f}%")

## Feature Selection

In [ ]:
# Polars missing value analysis
missing_df = (
    pl.DataFrame({
        "column": accepted_df.columns,
        "null_count": [accepted_df[col].is_null().sum() for col in accepted_df.columns]
    })
    .with_columns(
        (pl.col("null_count") / len(accepted_df) * 100).alias("null_pct")
    )
    .filter(pl.col("null_pct") > 0)
    .sort("null_pct", descending=True)
)

# Summary statistics
over_50 = missing_df.filter(pl.col("null_pct") >  50)
between_20 = missing_df.filter(
                (pl.col("null_pct") >  20) &
                (pl.col("null_pct") <= 50)
             )

print(f"Number of features with > 50% missing: {over_50.height}")
print(f"Features with 20–50% missing: {between_20.height}")
print(f"Selected columns which is < 20% missing values:")
print(accepted_df.select([col for col in accepted_df.columns
                          if (col not in over_50["column"].to_list())
                          & (col not in between_20["column"].to_list())]).columns)

### Build heatmap - correlated features

In [ ]:
FEATURES = [
    # Loan info
    'loan_amnt', 'term', 'int_rate', 'total_pymnt',
    'installment', 'grade', 'sub_grade', 'purpose',
    'funded_amnt', 'funded_amnt_inv',
     # Borrower financials
    'annual_inc', 'dti', 'delinq_2yrs', 'inq_last_6mths',
    'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
    # Employment
    'emp_length', 'home_ownership',
    # Credit history
    'earliest_cr_line', 'fico_range_low', 'fico_range_high',
    'last_pymnt_amnt', 'tot_cur_bal', 'total_rev_hi_lim',
    # Risk
    'verification_status', 'pub_rec_bankruptcies', 'tax_liens',
    'acc_now_delinq', 'mort_acc', 'bc_util', 'percent_bc_gt_75',
    # Target
    'issue_d', 'target'
]

# Filter to only existing columns (safety check)
available = [col for col in FEATURES if col in accepted_df.columns]
df = accepted_df.select(available)
print(f"Selected {len(available) - 1} features + target")
print(f"   Shape: {df.shape}")
print(df.null_count())

In [ ]:
def feat_corr(df: pl.DataFrame, features: list) -> None:
    """
    Computes and plots a correlation heatmap for all numeric features.

    Args:
        df       : Polars DataFrame
        features : list of feature column names
    """

    # Select only numeric columns
    numeric_df = df.select(features).select(cs.numeric()).to_pandas()
    non_numeric = [c for c in features if c not in numeric_df.columns]

    print(f"Numeric features   : {len(numeric_df.columns)}")
    print(f"Non-numeric features)  : {non_numeric}")

    # Compute correlation matrix
    corr_matrix = numeric_df.corr()

    # Plot
    n = len(corr_matrix)
    fig, ax = plt.subplots(figsize=(n * 0.7 + 2, n * 0.6 + 2))

    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))   # upper triangle mask

    sns.heatmap(
        corr_matrix,
        mask=mask,
        annot=True,
        fmt=".2f",
        annot_kws={"size": 7},
        cmap="RdBu_r",
        center=0,
        vmin=-1, vmax=1,
        linewidths=0.4,
        linecolor="white",
        square=True,
        cbar_kws={"shrink": 0.6, "label": "Pearson r"},
        ax=ax
    )

    ax.set_title("Feature Correlation Matrix (numeric only)",
                 fontsize=14, fontweight="bold", pad=15)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)

    plt.tight_layout()
    plt.savefig("../img/correlation_matrix.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Top 10 correlated pairs (excluding self-correlation)
    corr_pairs = (
        corr_matrix
        .unstack()
        .reset_index()
        .rename(columns={"level_0": "feature_1", "level_1": "feature_2", 0: "correlation"})
        .query("feature_1 != feature_2")                        # drop self
        .assign(abs_corr=lambda x: x["correlation"].abs())
        .drop_duplicates(subset=["abs_corr"])                   # drop mirror pairs
        .sort_values("abs_corr", ascending=False)
        .head(10)
    )

    print("\nTop 10 most correlated pairs:")
    print(corr_pairs[["feature_1", "feature_2", "correlation"]].to_string(index=False))

In [ ]:
# Feature correlation
feat_corr(df, FEATURES)

In [ ]:
(
    df
    .null_count()
    .transpose(include_header=True, header_name="column")
    .rename({"column_0": "null_count"})
    .filter(pl.col("null_count") > 0)
    .sort("null_count", descending=True)
)

### Final features + derived

In [ ]:
accepted_df["total_pymnt"].null_count()

In [ ]:
FEATURES_FINAL = [
    # Loan info
    'loan_amnt', 'term', 'int_rate', "total_pymnt",
    'grade', 'sub_grade', 'purpose', 'funded_amnt', 'funded_amnt_inv',
    # Borrower financials
    'annual_inc', 'dti', 'delinq_2yrs', 'inq_last_6mths',
    'open_acc', 'total_acc', 'pub_rec', 'revol_bal', 'revol_util',
    # Employment
    'emp_length', 'home_ownership',
    # Credit history
    'earliest_cr_line', 'fico_range_low',
    'last_pymnt_amnt', 'tot_cur_bal',
    # Risk
    'verification_status', 'pub_rec_bankruptcies', 'tax_liens',
    'acc_now_delinq', 'mort_acc', 'bc_util', 'percent_bc_gt_75',
    # Target
    'issue_d', 'target'
]

In [ ]:
clean_df = accepted_df.select(FEATURES_FINAL)

# Merge rare home_ownership
clean_df = clean_df.with_columns(
    pl.col('home_ownership').replace({'NONE': 'OTHER', 'ANY': 'OTHER'})
)

# extract integer in sub_grade
clean_df = clean_df.with_columns(
    pl.col("sub_grade").str.extract(r'(\d+)').cast(pl.Int8)
  )
# emp_length → integer
clean_df = clean_df.with_columns(
    pl.col('emp_length')
    .str.replace('10\\+ years', '10 years')
    .str.replace('< 1 year', '0 years')
    .str.extract(r'(\d+)')
    .cast(pl.Int8)
    .alias('emp_length')
)

# Fill missing values

# Fill missing values
clean_df = clean_df.with_columns(
    [
        pl.col("emp_length").fill_null(
            pl.col("emp_length").median()
        ),
        pl.col("percent_bc_gt_75").fill_null(
            pl.col("percent_bc_gt_75").mean()
        ),
        pl.col("mort_acc").fill_null(
            pl.col("mort_acc").median()
        ),
        pl.col("dti").fill_null(
            pl.col("dti").mean()
        ),
        pl.col("bc_util").fill_null(
            pl.col("bc_util").mean()
        ),
        pl.col("pub_rec_bankruptcies").fill_null(0),
        pl.col("inq_last_6mths").fill_null(0),
    ]
)

# Derived features
clean_df = clean_df.with_columns([
    # Loan-to-Income Ratio
    (pl.col("loan_amnt") / (pl.col("annual_inc") + 1))
        .alias("loan_to_income"),

    # Active Accounts Ratio
    (pl.col("open_acc") / (pl.col("total_acc") + 1))
        .alias("active_accounts_ratio"),

    # Inquiries per Account
    (pl.col("inq_last_6mths") / (pl.col("total_acc") + 1))
        .alias("inquiries_per_account"),

    # Payment-to-Loan Ratio
    (pl.col("total_pymnt") / (pl.col("loan_amnt") + 1))
        .alias("payment_to_loan"),
        
    # Revolving debt relative to income
    (pl.col("revol_bal") / (pl.col("annual_inc") + 1))
        .alias("revolbal_to_income")
])

In [ ]:
print(f"Data shape: {clean_df.shape}")     
print(f"Number of null columns: {clean_df.null_count()}")

In [ ]:
null_summary = (
    clean_df
    .null_count()
    .transpose(
        include_header=True,
        header_name="column",
        column_names=["null_count"]
    )
    .filter(pl.col("null_count") > 0)
)

print(null_summary)

## Save cleaned file

In [ ]:
clean_df.write_parquet("./processed_data/processed_accepted_df.parquet")